# Standard pH Calibration  (pH 1 - 12)

Calibrate a glass pH electrode and convert measured EMF into pH.
The response is assumed **linear (Nernstian)**:

$$\text{EMF} = E_0 - S\cdot \text{pH}$$

Choose one of two options:

- **Option 1 - pH at a SPECIFIC temperature.** Calibrate with buffers measured at one
  temperature, then predict sample pH at that same temperature.
- **Option 2 - pH at ANY temperature.** Calibrate with buffers measured at several
  temperatures; the notebook fits $E_0(T)$ and $S(T)$ so you can predict pH at any
  temperature in the calibrated range.

All input is via **inline arrays** - just edit the values in the marked cells and run.

*From Saleesongsom et al., ACS Omega (2026).*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats

np.set_printoptions(suppress=True)


---
## Option 1 - pH at a specific temperature

### 1a. Build the calibration
Enter buffer pH / EMF pairs measured at a single temperature (>= 2 points, 3+ better).


In [ ]:
# ===== EDIT YOUR NIST/DIN BUFFER DATA (single temperature) =====
# Standard pH calibration uses NIST/DIN buffers at pH 4, 7 and 10.
buffer_pH  = [4.009, 7.012, 10.015]   # NIST/DIN buffer pH values
buffer_EMF = [181.5, 5.9, -170.9]     # measured EMF (mV)
T_C        = 24.4                      # measurement temperature (deg C)
# ===============================================================

buffer_pH  = np.asarray(buffer_pH,  float)
buffer_EMF = np.asarray(buffer_EMF, float)

slope, intercept, r, p, se = stats.linregress(buffer_pH, buffer_EMF)
S1, E0_1, r2_1 = -slope, intercept, r**2
S_theory = 2.302585 * 8.314462 * (T_C + 273.15) / 96485.0 * 1000.0  # mV/pH

print(f"Calibration @ {T_C} C:  EMF = {E0_1:.2f} - {S1:.3f} * pH")
print(f"  slope S = {S1:.3f} mV/pH   E0 = {E0_1:.2f} mV   R^2 = {r2_1:.5f}")
print(f"  theoretical Nernst slope = {S_theory:.3f} mV/pH  ({100*S1/S_theory:.1f}% Nernstian)")

fig, ax = plt.subplots(figsize=(4, 3.2), dpi=120)
xl = np.array([0, 14])
ax.plot(xl, E0_1 - S1 * xl, "-", color="#E8261C", lw=1.6,
        label=f"EMF = {E0_1:.1f} - {S1:.2f}·pH\n$R^2$ = {r2_1:.4f}")
ax.scatter(buffer_pH, buffer_EMF, s=70, c="#F5A623", edgecolors="black", zorder=3)
ax.set_xlim(0, 14); ax.set_xlabel("pH"); ax.set_ylabel("EMF (mV)")
ax.legend(fontsize=8, frameon=False); ax.set_title(f"Standard calibration @ {T_C} °C")
plt.tight_layout(); plt.show()


### 1b. Predict sample pH
Enter the EMF of your sample(s) measured at the same temperature.
$$\text{pH} = (E_0 - \text{EMF})/S$$


In [ ]:
# ============ EDIT YOUR SAMPLE EMF (same temperature) ============
sample_EMF = [120.0, 30.0, -60.0]     # measured EMF (mV)
# =================================================================

sample_EMF = np.atleast_1d(np.asarray(sample_EMF, float))
sample_pH  = (E0_1 - sample_EMF) / S1

print(f"Predicted pH @ {T_C} C:")
for e, ph in zip(sample_EMF, sample_pH):
    print(f"  EMF {e:8.2f} mV  ->  pH {ph:6.3f}")


---
## Option 2 - pH at any temperature (temperature-corrected)

This follows the paper's construction (eqs 13-14; Figure 3a and SI Figure S12).
Standard 3-point calibrations are run at several temperatures to extract the
Nernstian slope $S$ and standard potential $E_0$ at each temperature. Their
temperature dependence is then obtained by **linear regression**:

$$S(T) = K_s\,(T+273.15) + C \qquad\text{(slope regressed on the Kelvin scale, eq 13)}$$

$$E_0(T) = E_{0,25} + \dfrac{dE_0}{dT}\,(T-25) \qquad\text{(intercept centered at 25 °C, eq 14)}$$

giving the practical temperature-dependent Nernst response

$$\text{EMF}(\text{pH},T) = E_0(T) - S(T)\cdot\text{pH}.$$

The paper values are $K_s = 0.199$ mV·°C⁻¹·pH⁻¹, $C = -0.54$ mV·pH⁻¹,
$E_{0,25} = 418.9$ mV, $dE_0/dT = 1.13$ mV·°C⁻¹.

### 2a. Enter standard calibrations at several temperatures
NIST/DIN buffers (pH 4, 7, 10) measured at each temperature.
The demo is seeded from the paper's 6.6-57.8 °C dataset.


In [ ]:
# ===== EDIT YOUR NIST/DIN BUFFER DATA (multiple temperatures) =====
# Standard 3-point calibration (pH 4, 7, 10) measured at each temperature.
# Demo seeded from the paper's 6.6-57.8 C dataset.
# Format:  temperature_C : ( [buffer pH], [measured EMF in mV] )
calibration_data = {
    6.81: ([4.01, 7.01, 10.01], [178.0, 12.2, -153.5]),
    15.889: ([4.01, 7.01, 10.01], [179.0, 9.0, -161.1]),
    24.411: ([4.01, 7.01, 10.01], [182.0, 5.8, -170.4]),
    31.5: ([4.01, 7.01, 10.01], [185.2, 4.3, -176.6]),
    38.889: ([4.01, 7.01, 10.01], [187.8, 2.2, -183.4]),
    48.311: ([4.01, 7.01, 10.01], [191.3, 0.8, -189.6]),
    57.8: ([4.01, 7.01, 10.01], [193.7, -1.6, -197.0]),
}
# =================================================================

In [ ]:
# ---- Step 1: extract S and E0 from a Nernst plot at each temperature ----
temps = np.array(sorted(calibration_data), float)
S_arr, E0_arr = [], []
for T in temps:
    ph, emf = map(lambda a: np.asarray(a, float), calibration_data[T])
    sl, ic, r, *_ = stats.linregress(ph, emf)
    S_arr.append(-sl); E0_arr.append(ic)
    print(f"T = {T:5.1f} C   S = {-sl:6.3f} mV/pH   E0 = {ic:7.2f} mV   R^2 = {r**2:.5f}")
S_arr, E0_arr = np.asarray(S_arr), np.asarray(E0_arr)

# ---- Step 2: temperature models (paper eqs 13-14) ----
# S(T)  = Ks*(T+273.15) + C      -> regress S on Kelvin
# E0(T) = E0_25 + dE0dT*(T-25)   -> regress E0 on (T-25)
cS = np.polyfit(temps + 273.15, S_arr, 1);  Ks, C = cS
cE = np.polyfit(temps - 25.0,  E0_arr, 1);  dE0dT, E0_25 = cE
S_of_T  = lambda T: Ks * (T + 273.15) + C
E0_of_T = lambda T: E0_25 + dE0dT * (T - 25.0)

def _r2(y, yh): return 1 - np.sum((y - yh)**2) / np.sum((y - np.mean(y))**2)
print(f"\nS(T)  = {Ks:.4f}·(T+273.15) + ({C:+.3f})       R^2 = {_r2(S_arr,  S_of_T(temps)):.4f}")
print(f"E0(T) = {E0_25:.2f} + ({dE0dT:+.4f})·(T-25)      R^2 = {_r2(E0_arr, E0_of_T(temps)):.4f}")


In [ ]:
# ---- Two regression graphs (S vs T and E0 vs T) + EMF(pH,T) family ----
def line_ci(xg, coeffs, xd, yd):
    '''95% confidence band on a degree-1 fit; returns (yhat, halfwidth).'''
    n = len(xd); dof = max(n - 2, 1)
    yhat = np.polyval(coeffs, xg)
    s2 = np.sum((yd - np.polyval(coeffs, xd))**2) / dof
    xbar = xd.mean(); Sxx = np.sum((xd - xbar)**2)
    se = np.sqrt(s2 * (1.0 / n + (xg - xbar)**2 / Sxx))
    return yhat, stats.t.ppf(0.975, dof) * se

Tg = np.linspace(temps.min(), temps.max(), 200)
fig, axs = plt.subplots(1, 3, figsize=(12, 3.2), dpi=120)

# (1) slope S(T)  -- regression on the Kelvin scale
yh, hw = line_ci(Tg + 273.15, cS, temps + 273.15, S_arr)
axs[0].fill_between(Tg, yh - hw, yh + hw, color="#ff9999", alpha=0.35, label="95% CI")
axs[0].plot(Tg, yh, "r-", lw=1.5, label="linear regression")
axs[0].scatter(temps, S_arr, s=45, c="0.5", edgecolors="k", zorder=3, label="data")
axs[0].set_xlabel("Temp (°C)"); axs[0].set_ylabel(r"Slope $S(T)$ (mV·pH$^{-1}$)")
axs[0].set_title("$S(T)$"); axs[0].legend(fontsize=7, frameon=False)

# (2) standard potential E0(T)  -- centered at 25 C
yh, hw = line_ci(Tg - 25.0, cE, temps - 25.0, E0_arr)
axs[1].fill_between(Tg, yh - hw, yh + hw, color="#ff9999", alpha=0.35)
axs[1].plot(Tg, yh, "r-", lw=1.5)
axs[1].scatter(temps, E0_arr, s=45, c="0.5", edgecolors="k", zorder=3)
axs[1].set_xlabel("Temp (°C)"); axs[1].set_ylabel(r"Standard potential $E_0(T)$ (mV)")
axs[1].set_title("$E_0(T)$")

# (3) EMF(pH, T) family
pH_grid = np.linspace(1, 12, 200)
cmap = plt.cm.autumn_r; norm = mpl.colors.Normalize(temps.min(), temps.max())
for T in np.linspace(temps.min(), temps.max(), 40):
    axs[2].plot(pH_grid, E0_of_T(T) - S_of_T(T) * pH_grid, color=cmap(norm(T)), lw=0.9)
axs[2].set_xlim(1, 12); axs[2].set_xlabel("pH"); axs[2].set_ylabel("EMF (mV)")
axs[2].set_title("EMF(pH, T)")
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=axs[2], pad=0.02).set_label("Temp (°C)")
plt.tight_layout(); plt.show()


### 2b. Predict sample pH at any temperature
Enter each sample EMF **and** its temperature.
$$\text{pH} = \dfrac{E_0(T)-\text{EMF}}{S(T)}$$


In [ ]:
# ============ EDIT YOUR SAMPLE MEASUREMENTS ============
sample_EMF = [150.0, 0.0, -150.0]     # measured EMF (mV)
sample_T   = [20.0, 30.0, 35.0]       # temperature of each sample (deg C)
# ======================================================

sample_EMF = np.atleast_1d(np.asarray(sample_EMF, float))
sample_T   = np.atleast_1d(np.asarray(sample_T, float))
if sample_T.size == 1:
    sample_T = np.full_like(sample_EMF, sample_T[0])

print("EMF (mV) @ T (C)  ->  predicted pH")
for e, T in zip(sample_EMF, sample_T):
    ph = (E0_of_T(T) - e) / S_of_T(T)
    warn = "  (T outside calibrated range)" if (T < temps.min() or T > temps.max()) else ""
    print(f"  {e:8.2f} @ {T:5.1f}  ->  pH {ph:6.3f}{warn}")
